In [11]:
import os
import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)

from dotenv import load_dotenv

# Prefer the new standalone packages; fall back to langchain_community if not installed
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load the API key from the .env file in this folder
load_dotenv(override=True)
api_key = os.getenv("GROQ_API_KEY")
print("key loaded:", bool(api_key), "| length:", len(api_key) if api_key else 0)
assert api_key, "GROQ_API_KEY is empty. Check the .env file (one line: GROQ_API_KEY=your_key)"

DB_PATH = "./chroma_db"

key loaded: True | length: 56


In [12]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
print(f"Vector store loaded. Total vectors: {vectorstore._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store loaded. Total vectors: 3093


In [13]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=api_key,
    temperature=0.2
)

In [14]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [15]:
prompt = ChatPromptTemplate.from_template("""You are a helpful assistant answering questions based ONLY on the provided context.
If the answer is not in the context, say "I don't have enough information in my documents to answer this."
Do not make up information.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [16]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [17]:
def ask(question):
    print(f"\nQUESTION: {question}")
    answer = rag_chain.invoke(question)
    print(f"ANSWER: {answer}")
    docs = retriever.invoke(question)
    print("Sources:")
    seen = set()
    for doc in docs:
        source = os.path.basename(doc.metadata.get('source', 'unknown'))
        page = doc.metadata.get('page_label', '?')
        key = f"{source} (page {page})"
        if key not in seen:
            print(f"  - {key}")
            seen.add(key)
    return answer

In [21]:
ask("explain indian constitution")



QUESTION: explain indian constitution
ANSWER: **Explanation of the Indian Constitution (based on the provided documents)**  

The Constitution of India is the supreme law of the country, adopted on 26 November 1949. It sets out the structure, powers, and functions of the government and the rights of its citizens. The documents you supplied give a concise overview of its key elements:

| Section | What it contains (as per the documents) |
|---------|----------------------------------------|
| **Preamble** | Declares India to be a *sovereign, socialist, secular, democratic republic* and lists the fundamental values that the Constitution seeks to secure: justice, liberty, equality, fraternity, and the unity and integrity of the nation. |
| **Preface** | Notes that this is the sixth pocket‑size edition, updated to include all amendments up to the 106th Amendment Act, 2023. It also references additional documents (Appendix I–III) that provide details on territorial changes, the Constitutio

'**Explanation of the Indian Constitution (based on the provided documents)**  \n\nThe Constitution of India is the supreme law of the country, adopted on 26\u202fNovember\u202f1949. It sets out the structure, powers, and functions of the government and the rights of its citizens. The documents you supplied give a concise overview of its key elements:\n\n| Section | What it contains (as per the documents) |\n|---------|----------------------------------------|\n| **Preamble** | Declares India to be a *sovereign, socialist, secular, democratic republic* and lists the fundamental values that the Constitution seeks to secure: justice, liberty, equality, fraternity, and the unity and integrity of the nation. |\n| **Preface** | Notes that this is the sixth pocket‑size edition, updated to include all amendments up to the 106th Amendment Act,\u202f2023. It also references additional documents (Appendix\u202fI–III) that provide details on territorial changes, the Constitution (Application to J